# Sentiment-Driven Nifty Volatility Prediction — v2

Rebuild after a critical review of v1's backtest. The full diagnosis is in
`CRITICAL_ANALYSIS.md`; the three-line version:

- v1 sized positions by dividing a target vol by a **Garman-Klass** forecast,
  which measures intraday vol only. The traded return is close-to-close and
  includes the overnight gap → every position ~1.24x too big.
- `exp(mean of log vol)` is the median, not the mean → another ~1.09x.
- Daily rebalancing off a noisy daily forecast at 7.5 bps cost ~3.5%/yr.

Predicted over-leverage 1.35x; leverage backed out of v1's own reported
results 1.31x. That accounts for the loss to buy-and-hold.

**And fixing it is not enough.** Sharpe is invariant to leverage, so a
long-only index position sized by a vol forecast has no expected-return edge.
The forecast has to be traded against a *price* for volatility — **India VIX**
— which is what section 10 does.

## 0. Setup

In [ ]:
!pip install -q feedparser optuna shap arch lightgbm yfinance --upgrade

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta

pd.set_option("display.width", 120)
RANDOM_SEED = 42

## 1. Volatility estimators and unit conventions

The single most important cell in the notebook. There are two different daily
volatilities and v1 conflated them: **intraday** (what Garman-Klass measures)
and **total close-to-close** (what you actually trade). Everything downstream
works in total units, which is also what India VIX quotes in — that is what
makes the two comparable and the VRP strategy possible.

In [ ]:
# ===== src/volatility.py =====
"""Volatility estimators, and the unit conventions the rest of the pipeline depends on.

The single most important idea in this module is that there are *two different*
daily volatilities and v1 of this project conflated them:

  * **Intraday (open-to-close) vol** -- what Garman-Klass measures. It is built
    from that session's O/H/L/C and knows nothing about the gap between
    yesterday's close and today's open.
  * **Total (close-to-close) vol** -- what you actually trade. A position held
    overnight earns ``log(C_t / C_{t-1})``, whose variance includes the
    overnight gap.

For the Nifty the overnight gap is a large minority of total daily variance
(roughly a third), so ``sigma_GK`` systematically understates the risk of a
close-to-close position by ~20%. Sizing a position as
``target_vol / predicted_GK_vol`` therefore over-levers by ~1/0.8 = 1.25x
*before* anything else goes wrong. See CRITICAL_ANALYSIS.md section 1.

Everything downstream of this module works in **total** vol units, which are
also the units India VIX quotes in -- that is what makes the two comparable
and the variance-risk-premium strategy possible at all.
"""


import numpy as np
import pandas as pd

TRADING_DAYS = 252
_GK_CO_COEF = 2.0 * np.log(2.0) - 1.0


def garman_klass_variance(df: pd.DataFrame) -> pd.Series:
    """Garman-Klass *intraday* variance from one session's own OHLC.

        sigma^2_GK = 0.5 * ln(H/L)^2 - (2 ln2 - 1) * ln(C/O)^2

    Floored at zero: the estimator is unbiased but not guaranteed positive on
    any single observation.
    """
    log_hl = np.log(df["high"] / df["low"])
    log_co = np.log(df["close"] / df["open"])
    return (0.5 * log_hl**2 - _GK_CO_COEF * log_co**2).clip(lower=0.0)


def gap_variance(df: pd.DataFrame) -> pd.Series:
    """Overnight variance contribution, ``ln(O_t / C_{t-1})^2``.

    A one-observation estimator of the overnight component, exactly analogous
    to using ``r_t^2`` as a one-observation estimator of daily variance: noisy
    per day, unbiased in expectation, which is all we need since it is being
    summed into a target that gets modelled in logs.
    """
    return np.log(df["open"] / df["close"].shift(1)) ** 2


def total_daily_variance(df: pd.DataFrame) -> pd.Series:
    """Gap-inclusive daily variance = overnight variance + intraday variance.

    This is the decomposition of a close-to-close return into its two legs. It
    is the correct target for anything that sizes or prices an *overnight*
    position, and it is directly comparable to India VIX (de-annualized).
    """
    return gap_variance(df) + garman_klass_variance(df)


def realized_vol(df: pd.DataFrame, kind: str = "total") -> pd.Series:
    """Daily volatility (not variance, not annualized) for the given estimator."""
    if kind == "total":
        var = total_daily_variance(df)
    elif kind == "gk":
        var = garman_klass_variance(df)
    elif kind == "close":
        var = np.log(df["close"] / df["close"].shift(1)) ** 2
    else:
        raise ValueError(f"unknown estimator kind: {kind!r}")
    return np.sqrt(var)


def annualize(daily_vol: pd.Series | float, periods: int = TRADING_DAYS):
    return daily_vol * np.sqrt(periods)


def deannualize(annual_vol: pd.Series | float, periods: int = TRADING_DAYS):
    return annual_vol / np.sqrt(periods)


# --------------------------------------------------------------------------
# log-space -> level-space conversion
# --------------------------------------------------------------------------

def smearing_factor(log_residuals: np.ndarray | pd.Series) -> float:
    """Duan's smearing estimate of ``E[exp(residual)]``.

    Models are fit on ``log`` volatility because volatility is right-skewed and
    strictly positive. But ``exp(E[log v])`` is the *median*, not the mean --
    it understates the level by roughly ``exp(sigma_resid^2 / 2)``. With the
    residual spread this model actually achieves (~0.41 in log units) that is
    an 8-9% low bias, which feeds straight into over-leverage when you divide
    by it.

    Duan's smearing estimator is the non-parametric version of that correction
    and does not assume the residuals are Gaussian. It MUST be estimated on
    training residuals only.
    """
    resid = np.asarray(log_residuals, dtype=float)
    resid = resid[np.isfinite(resid)]
    if resid.size == 0:
        return 1.0
    return float(np.mean(np.exp(resid)))


def log_to_level(log_pred: pd.Series | np.ndarray, smearing: float = 1.0):
    """Convert a log-vol forecast to a vol level, applying the mean correction."""
    return np.exp(log_pred) * smearing

## 2. Data

`^NSEI` for prices and **`^INDIAVIX` for implied volatility**. The VIX was
missing from v1 entirely and is the biggest single omission in the project:
it is a forward-looking, market-consensus volatility forecast, published
daily and free, and it already impounds most of what news sentiment could
carry.

`simulate_market` is here so the notebook runs end to end offline.

In [ ]:
# ===== src/data.py =====
"""Data loading, plus a synthetic market for testing without network access.

**On sourcing the data.** yfinance is the convenient path and usually works,
but Yahoo rate-limits datacenter IP ranges hard, and Kaggle/Colab runners sit
squarely in those ranges. A run can fail with 429 or an empty frame on a day
when the same call works fine from a laptop. So the loader tries several
sources in order of reliability rather than assuming one:

1. **Local CSV** -- a Kaggle dataset mounted into the notebook. No internet
   required at all, so it cannot be rate-limited. This is the only route that
   is guaranteed to work, and it is the one to use if a run matters.
2. **yfinance** -- convenient, usually fine, occasionally throttled.
3. **Stooq** -- a different host with a friendlier attitude to datacenter IPs.
   It carries the Nifty index but *not* India VIX.

India VIX is the dependency that actually matters. Without it there is no
implied-vol series, and without that the variance-risk-premium strategy -- the
only leg with a real expected-return edge -- cannot run at all. The vol
forecasting and the directional book still work on price data alone.

``simulate_market`` generates a series with the structural properties this
project depends on -- volatility clustering, a leverage effect, a realistic
overnight-gap share of variance, and an implied vol carrying a positive
variance risk premium -- so the pipeline can be exercised end to end offline.
Synthetic results verify that the *code* is correct. They say nothing about
whether the strategy works on the Nifty.
"""


import pathlib

import numpy as np
import pandas as pd


TRADING_DAYS = 252

_OHLC_ALIASES = {
    "date": "date", "datetime": "date", "timestamp": "date", "time": "date",
    "open": "open", "high": "high", "low": "low",
    "close": "close", "adj close": "close", "adj_close": "close",
    "price": "close", "ltp": "close", "closing value": "close",
    "shares traded": "volume", "volume": "volume", "vol": "volume",
    "turnover": "turnover",
}


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Map the many spellings CSV exports use onto our canonical names.

    NSE's own exports, Kaggle datasets and Yahoo CSVs all differ; normalising
    here is cheaper than making every caller guess.
    """
    out = df.copy()
    out.columns = [_OHLC_ALIASES.get(str(c).strip().lower(), str(c).strip().lower())
                   for c in out.columns]
    return out.loc[:, ~out.columns.duplicated()]


def load_ohlc_csv(path: str | pathlib.Path) -> pd.DataFrame:
    """Read an OHLC CSV with tolerant column naming and a parsed date index."""
    df = _normalize_columns(pd.read_csv(path))
    if "date" not in df.columns:
        raise ValueError(f"{path}: no recognisable date column in {list(df.columns)}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["date"]).set_index("date").sort_index()
    df.index = df.index.tz_localize(None) if df.index.tz is not None else df.index
    df.index.name = "date"

    missing = {"open", "high", "low", "close"} - set(df.columns)
    if missing:
        raise ValueError(f"{path}: missing {sorted(missing)}")

    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            # Indian exports frequently carry thousands separators.
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(",", "", regex=False),
                errors="coerce")
    if "volume" not in df.columns:
        df["volume"] = np.nan

    return df[["open", "high", "low", "close", "volume"]].dropna(
        subset=["open", "high", "low", "close"])


def load_vix_csv(path: str | pathlib.Path) -> pd.Series:
    """Read an India VIX CSV; returns the closing level in annualized percent."""
    df = load_ohlc_csv(path) if _has_ohlc(path) else _load_single_series_csv(path)
    s = df["close"] if isinstance(df, pd.DataFrame) else df
    return s.rename("vix").dropna()


def _has_ohlc(path) -> bool:
    cols = set(_normalize_columns(pd.read_csv(path, nrows=1)).columns)
    return {"open", "high", "low", "close"}.issubset(cols)


def _load_single_series_csv(path) -> pd.Series:
    df = _normalize_columns(pd.read_csv(path))
    if "date" not in df.columns:
        raise ValueError(f"{path}: no recognisable date column")
    value_cols = [c for c in df.columns if c != "date"]
    if not value_cols:
        raise ValueError(f"{path}: no value column")
    df["date"] = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["date"]).set_index("date").sort_index()
    return pd.to_numeric(
        df[value_cols[-1]].astype(str).str.replace(",", "", regex=False),
        errors="coerce")


def _from_yfinance(ticker: str, years: int):
    try:
        import yfinance as yf
        df = yf.download(ticker, period=f"{years}y", interval="1d",
                         auto_adjust=False, progress=False, threads=False)
    except Exception:
        return None
    if df is None or len(df) == 0:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.rename(columns=str.lower)
    if "close" not in df.columns:
        return None
    if "volume" not in df.columns:
        df["volume"] = np.nan
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "date"
    keep = [c for c in ["open", "high", "low", "close", "volume"] if c in df.columns]
    return df[keep].dropna(subset=["close"])


def _from_stooq(candidates: tuple[str, ...], years: int):
    """Stooq via pandas-datareader. Carries the Nifty index; not India VIX."""
    try:
        from pandas_datareader import data as pdr
    except Exception:
        return None
    start = pd.Timestamp.today() - pd.DateOffset(years=years)
    for sym in candidates:
        try:
            df = pdr.DataReader(sym, "stooq", start=start)
        except Exception:
            continue
        if df is None or len(df) < 200:
            continue
        df = df.rename(columns=str.lower).sort_index()
        if "volume" not in df.columns:
            df["volume"] = np.nan
        df.index = pd.to_datetime(df.index).tz_localize(None)
        df.index.name = "date"
        return df[["open", "high", "low", "close", "volume"]].dropna(
            subset=["open", "high", "low", "close"])
    return None


def load_market_data(price_ticker: str = "^NSEI",
                     vix_ticker: str = "^INDIAVIX",
                     years: int = 10,
                     price_csv: str | pathlib.Path | None = None,
                     vix_csv: str | pathlib.Path | None = None,
                     verbose: bool = True) -> tuple[pd.DataFrame, pd.Series | None, dict]:
    """Load Nifty OHLC and India VIX, trying CSV then yfinance then Stooq.

    Returns ``(prices, vix, provenance)``. ``vix`` is None when no implied-vol
    series could be sourced -- the caller must handle that rather than get a
    silently empty column, because it decides whether the VRP strategy can run.
    """
    prov = {}

    prices = None
    if price_csv and pathlib.Path(price_csv).exists():
        prices = load_ohlc_csv(price_csv)
        prov["prices"] = f"csv:{price_csv}"
    if prices is None:
        prices = _from_yfinance(price_ticker, years)
        if prices is not None:
            prov["prices"] = f"yfinance:{price_ticker}"
    if prices is None:
        prices = _from_stooq(("^NSEI", "^nsei"), years)
        if prices is not None:
            prov["prices"] = "stooq:^NSEI"
    if prices is None:
        raise RuntimeError(
            "Could not load Nifty prices from CSV, yfinance or Stooq. On Kaggle: "
            "check Settings -> Internet is On, or attach a Nifty dataset and pass "
            "price_csv=<path>.")

    vix = None
    if vix_csv and pathlib.Path(vix_csv).exists():
        vix = load_vix_csv(vix_csv)
        prov["vix"] = f"csv:{vix_csv}"
    if vix is None:
        v = _from_yfinance(vix_ticker, years)
        if v is not None:
            vix = v["close"].rename("vix")
            prov["vix"] = f"yfinance:{vix_ticker}"
    if vix is None:
        prov["vix"] = "UNAVAILABLE"

    if verbose:
        print(f"prices <- {prov['prices']}: {len(prices)} sessions  "
              f"{prices.index.min().date()} -> {prices.index.max().date()}")
        if vix is None:
            print("vix    <- UNAVAILABLE. The VRP strategy cannot run without an "
                  "implied-vol series; forecasting and the directional book still can.")
        else:
            print(f"vix    <- {prov['vix']}: {len(vix)} sessions  mean {vix.mean():.1f}")

    return prices, vix, prov


# Backwards-compatible thin wrappers.
def load_prices(ticker: str = "^NSEI", years: int = 8) -> pd.DataFrame:
    df = _from_yfinance(ticker, years)
    if df is None:
        raise RuntimeError(f"yfinance returned nothing for {ticker}")
    return df


def load_india_vix(years: int = 8) -> pd.Series:
    df = _from_yfinance("^INDIAVIX", years)
    if df is None:
        raise RuntimeError("yfinance returned nothing for ^INDIAVIX")
    return df["close"].rename("vix")


def simulate_market(n_days: int = 2200, seed: int = 7,
                    gap_share: float = 0.35,
                    vrp: float = 0.15,
                    drift_annual: float = 0.10,
                    jump_prob: float = 0.005,
                    jump_size: float = 1.0,
                    intraday_steps: int = 200,
                    start: str = "2016-01-04") -> tuple[pd.DataFrame, pd.Series]:
    """GJR-GARCH-like synthetic index with OHLC and a matching implied 

    Three properties matter, because the tests and the demo depend on them:

    * The close-to-close return is ``gap + intraday``, with the legs drawn
      independently so ``var(gap) = gap_share * var(total)`` holds exactly.
    * High and low come from a simulated intraday path, not a made-up range,
      so Garman-Klass is an unbiased estimator of the intraday leg. Otherwise
      the tests would be checking the simulator's arithmetic.
    * **Implied vol is causal.** It is built from an EWMA of past returns --
      what a market maker could actually know at ``t`` -- marked up by a
      persistent, time-varying premium that sometimes goes negative. An IV
      built from the *realized* forward vol would hand the VRP strategy a
      risk-free arbitrage and print a meaningless Sharpe.

    Parameters
    ----------
    gap_share
        Fraction of total daily variance arriving overnight. ~0.35 is in the
        right range for the Nifty and is exactly the quantity that makes a
        Garman-Klass target the wrong denominator for position sizing.
    vrp
        Mean level of the variance risk premium: implied vol averages
        ``(1 + vrp)`` times the market maker's own vol estimate.
    jump_prob, jump_size
        Poisson variance jumps. A vol process without jumps has no left tail,
        which would delete precisely the risk that defines short-volatility
        trading and make the VRP backtest print a fantasy Sharpe. These are
        what stop the premium from looking free.
    """
    rng = np.random.default_rng(seed)

    omega, alpha, gamma, beta = 3.0e-6, 0.05, 0.08, 0.88
    mu = drift_annual / TRADING_DAYS
    var = np.empty(n_days)
    ret = np.empty(n_days)
    gap = np.empty(n_days)
    intra = np.empty(n_days)
    var[0] = omega / (1 - alpha - gamma / 2 - beta)

    t_scale = np.sqrt(6 / 4)  # unit-variance Student-t(6)
    for t in range(n_days):
        if t > 0:
            shock = ret[t - 1] - mu
            var[t] = (omega + alpha * shock**2
                      + gamma * shock**2 * (shock < 0) + beta * var[t - 1])
            if rng.random() < jump_prob:
                var[t] *= 1.0 + rng.exponential(jump_size)
        sd = np.sqrt(var[t])
        # the drift is carried in the overnight leg, as it is in practice
        gap[t] = mu + sd * np.sqrt(gap_share) * rng.standard_normal()
        intra[t] = sd * np.sqrt(1 - gap_share) * rng.standard_t(df=6) / t_scale
        ret[t] = gap[t] + intra[t]

    sigma = np.sqrt(var)
    intra_sd = sigma * np.sqrt(1 - gap_share)

    close = 10000 * np.exp(np.cumsum(ret))
    prev_close = np.concatenate([[10000.0], close[:-1]])
    open_ = prev_close * np.exp(gap)

    # Intraday path as a Brownian bridge pinned to the realized open->close
    # move, so H/L are consistent with the intraday variance by construction.
    m = intraday_steps
    steps = rng.standard_normal((n_days, m)) * (intra_sd[:, None] / np.sqrt(m))
    walk = np.cumsum(steps, axis=1)
    tgrid = np.arange(1, m + 1) / m
    bridge = walk - tgrid * walk[:, -1][:, None] + tgrid * intra[:, None]
    bridge = np.concatenate([np.zeros((n_days, 1)), bridge], axis=1)

    high = open_ * np.exp(bridge.max(axis=1))
    low = open_ * np.exp(bridge.min(axis=1))

    idx = pd.bdate_range(start=start, periods=n_days, name="date")
    prices = pd.DataFrame(
        {"open": open_, "high": high, "low": low, "close": close,
         "volume": rng.integers(1e5, 5e5, n_days)}, index=idx)

    # --- causal implied vol -------------------------------------------------
    # A market maker's own vol estimate: EWMA of squared returns up to t.
    lam = 0.94
    ewma = np.empty(n_days)
    ewma[0] = var[0]
    for t in range(1, n_days):
        ewma[t] = lam * ewma[t - 1] + (1 - lam) * (ret[t - 1] - mu) ** 2

    # Persistent, mean-reverting premium that dips negative after vol shocks.
    prem = np.empty(n_days)
    prem[0] = vrp
    for t in range(1, n_days):
        prem[t] = 0.97 * prem[t - 1] + 0.03 * vrp + rng.standard_normal() * 0.035
    prem = np.clip(prem, -0.25, 0.70)

    # Idiosyncratic, persistent quoting noise. Real implied vol is not a clean
    # function of trailing realized vol -- log-IV regressed on log trailing RV
    # leaves ~25-30% of the variance unexplained. Without that residual the
    # model can invert the premium exactly and the VRP backtest becomes an
    # arbitrage rather than a risk premium.
    iv_noise = np.empty(n_days)
    iv_noise[0] = 0.0
    for t_ in range(1, n_days):
        iv_noise[t_] = 0.85 * iv_noise[t_ - 1] + rng.standard_normal() * 0.06
    iv_ann = np.sqrt(ewma * TRADING_DAYS) * (1 + prem) * np.exp(iv_noise)
    vix = pd.Series(iv_ann * 100, index=idx, name="vix")
    return prices, vix


def simulate_news(index: pd.DatetimeIndex, prices: pd.DataFrame,
                  seed: int = 11, signal_strength: float = 0.35) -> pd.DataFrame:
    """Synthetic FinBERT-scored headlines with a weak genuine vol linkage.

    Article volume and sentiment intensity are tied to the *next* day's
    volatility with ``signal_strength``, so a working pipeline should be able
    to extract something. Set ``signal_strength=0`` to confirm the model
    correctly finds nothing.
    """
    rng = np.random.default_rng(seed)
    rv = realized_vol(prices, "total").reindex(index)
    z = ((np.log(rv + 1e-8) - np.log(rv + 1e-8).mean())
         / np.log(rv + 1e-8).std()).fillna(0.0)
    z_fwd = z.shift(-1).fillna(0.0)

    rows = []
    for day in index:
        drive = signal_strength * z_fwd.loc[day] + rng.standard_normal() * 0.5
        n = max(1, int(np.exp(3.0 + 0.35 * drive + rng.standard_normal() * 0.25)))
        intensity = np.clip(0.35 + 0.15 * drive, 0.05, 0.95)
        pol = np.clip(rng.standard_normal(n) * intensity, -1, 1)
        pos = np.clip(0.5 + pol / 2, 0, 1)
        rows.append(pd.DataFrame({
            "trading_day": day,
            "polarity": pol,
            "positive": pos,
            "negative": 1 - pos,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
prices = load_prices("^NSEI", years=8)
vix = load_india_vix(years=8)
print(prices.shape, prices.index.min().date(), "->", prices.index.max().date())
print("India VIX:", vix.shape, f"mean {vix.mean():.1f}")
prices.tail()

## 3. News collection and FinBERT scoring

Unchanged from v1 — GDELT DOC 2.0 for backfill, live RSS going forward,
dedupe, keyword relevance filter, trading-day bucketing on the
(prev close 15:30 IST, open 09:15 IST] window, then `ProsusAI/finbert`
inference to a `polarity = P(pos) - P(neg)` score per headline.

Paste those cells from the v1 notebook here. The only thing v2 needs out of
them is a DataFrame with columns `trading_day`, `polarity`, `positive`,
`negative`. If you don't have it yet, leave `scored_news = None` — the
pipeline runs price + VIX only.

In [ ]:
scored_news = None   # <- v1's FinBERT output goes here

## 4. Features

Three changes from v1, in descending order of impact:

1. **India VIX is a feature** (see above).
2. **Targets are gap-inclusive and multi-horizon.** `h=1` keeps the error
   metrics comparable with v1; `h=5` is what the weekly-expiry VRP trade needs.
   Two flavours per horizon: mean forward *log-vol* for modelling, and
   arithmetic forward *variance* for settling the variance swap.
3. **Sentiment measures intensity and surprise, not direction.** v1 fed a
   *signed* mean polarity to predict a *magnitude* — good and bad news both
   raise vol, so the sign is nearly irrelevant, and averaging ~50 headlines
   crushes the remaining variance. Missing days stay NaN instead of being
   filled with 0.0, which in v1 made "no news" indistinguishable from
   "perfectly balanced news".

In [ ]:
# ===== src/features.py =====
"""Feature construction.

Three changes from v1, in descending order of how much they matter:

1. **India VIX is now a feature.** Its absence was the single biggest miss in
   v1. Implied volatility is a forward-looking, market-consensus volatility
   forecast; it is the strongest known predictor of next-day realized vol and
   it subsumes most of what news sentiment could ever tell you. A vol model
   that does not use it is competing with one hand tied.

2. **Targets are gap-inclusive and multi-horizon.** See src/volatility.py for
   why. ``h=1`` keeps the error metrics comparable with v1; ``h=5`` is what the
   tradeable weekly-expiry strategy actually needs.

3. **Sentiment features measure intensity and surprise, not direction.**
   v1 fed the model ``mean(polarity)`` -- a *signed* quantity -- and asked it to
   predict a *magnitude*. Good news and bad news both raise volatility, so the
   sign is close to irrelevant and averaging ~50 headlines crushes what little
   variance remains. What actually moves volatility is how much is being
   written, how extreme it is, and how much the outlets disagree. Missing days
   are also left as NaN rather than filled with 0.0: in v1, "no news" and
   "perfectly balanced news" were encoded identically, which is a contaminated
   feature. LightGBM routes NaN natively.
"""


import numpy as np
import pandas as pd


EPS = 1e-8

PRICE_FEATURES = [
    "log_rv", "log_rv_w", "log_rv_m", "log_rv_q",
    "log_gk", "log_gap_share",
    "ret_1d", "ret_5d", "neg_ret_1d",
    "rv_trend", "rv_of_rv",
    "day_of_week",
]

VIX_FEATURES = [
    "log_iv", "log_vrp", "vix_chg_1d", "vix_chg_5d", "log_iv_slope",
]

SENTIMENT_FEATURES = [
    "sent_abs_mean", "sent_disp", "sent_tail_share", "sent_neg_share",
    "sent_skew", "news_vol_z", "sent_abs_shock", "sent_disp_roll3",
    "sent_abs_mean_roll5", "has_news",
]


# --------------------------------------------------------------------------
# price / autoregressive block
# --------------------------------------------------------------------------

def build_price_features(prices: pd.DataFrame) -> pd.DataFrame:
    """Realized-vol, HAR-style and return features, all known at date t's close."""
    out = prices.copy()

    out["log_return"] = np.log(out["close"] / out["close"].shift(1))
    out["rv"] = realized_vol(out, "total")      # gap-inclusive -- what we trade
    out["gk_vol"] = realized_vol(out, "gk")     # intraday only -- kept for reference
    out["log_rv"] = np.log(out["rv"] + EPS)
    out["log_gk"] = np.log(out["gk_vol"] + EPS)

    # Share of the day's variance that came from the overnight gap. Regime
    # information: gap-heavy periods are event-driven, range-heavy are grind.
    gk_var = garman_klass_variance(out)
    out["log_gap_share"] = np.log(
        ((out["rv"] ** 2 - gk_var).clip(lower=0) + EPS) / (out["rv"] ** 2 + EPS)
    )

    # HAR (Corsi 2009) cascade: daily / weekly / monthly / quarterly log-RV.
    out["log_rv_w"] = out["log_rv"].rolling(5).mean()
    out["log_rv_m"] = out["log_rv"].rolling(22).mean()
    out["log_rv_q"] = out["log_rv"].rolling(66).mean()

    out["ret_1d"] = out["log_return"]
    out["ret_5d"] = out["log_return"].rolling(5).sum()
    # Leverage effect: downside returns raise future vol far more than upside.
    out["neg_ret_1d"] = out["log_return"].clip(upper=0.0)

    out["rv_trend"] = out["log_rv_w"] - out["log_rv_m"]
    out["rv_of_rv"] = out["log_rv"].rolling(22).std()

    out["day_of_week"] = out.index.dayofweek
    return out


def add_vix_features(df: pd.DataFrame, vix_close: pd.Series | None) -> pd.DataFrame:
    """Join India VIX and derive the implied-vs-realized spread.

    ``vix_close`` is the India VIX closing level in annualized percent (the way
    it is quoted). The close at date t is known at t's close, so using it to
    forecast t+1 is not leakage.
    """
    out = df.copy()
    if vix_close is None or len(vix_close) == 0:
        for col in VIX_FEATURES:
            out[col] = np.nan
        out["iv_daily"] = np.nan
        return out

    vix = vix_close.reindex(out.index).ffill(limit=3)
    out["vix"] = vix
    out["iv_daily"] = deannualize(vix / 100.0)
    out["log_iv"] = np.log(out["iv_daily"] + EPS)

    # The variance risk premium in log space -- implied vol richness over
    # trailing realized. Strongly mean-reverting, and the core trading signal.
    out["log_vrp"] = out["log_iv"] - out["log_rv_w"]

    out["vix_chg_1d"] = out["log_iv"].diff()
    out["vix_chg_5d"] = out["log_iv"].diff(5)
    # Term-structure proxy: spot IV vs its own recent average.
    out["log_iv_slope"] = out["log_iv"] - out["log_iv"].rolling(22).mean()
    return out


# --------------------------------------------------------------------------
# sentiment block
# --------------------------------------------------------------------------

def aggregate_daily_sentiment(scored_news: pd.DataFrame) -> pd.DataFrame:
    """One row per trading day of *volatility-relevant* sentiment aggregates.

    Expects the columns produced by the v1 FinBERT step: ``trading_day``,
    ``polarity`` (= P(pos) - P(neg)), ``positive``, ``negative``.
    """
    cols = ["trading_day", "sent_mean", "sent_abs_mean", "sent_disp",
            "sent_tail_share", "sent_neg_share", "sent_skew", "article_count"]
    if scored_news is None or len(scored_news) == 0:
        return pd.DataFrame(columns=cols)

    news = scored_news.copy()
    news["abs_polarity"] = news["polarity"].abs()
    # "Tail" = an unambiguously charged headline. Averaging polarity over a day
    # hides these; their *count* is what tracks volatility.
    news["is_tail"] = (news["abs_polarity"] > 0.8).astype(float)
    news["is_negative"] = (news["negative"] > news["positive"]).astype(float)

    g = news.groupby("trading_day")
    daily = g.agg(
        sent_mean=("polarity", "mean"),
        sent_abs_mean=("abs_polarity", "mean"),
        sent_disp=("polarity", "std"),
        sent_tail_share=("is_tail", "mean"),
        sent_neg_share=("is_negative", "mean"),
        sent_skew=("polarity", "skew"),
        article_count=("polarity", "size"),
    ).reset_index()

    # std/skew are undefined for a 1-headline day; 0 dispersion is the honest
    # reading, but skew is genuinely unknown.
    daily["sent_disp"] = daily["sent_disp"].fillna(0.0)
    return daily[cols]


def add_sentiment_dynamics(daily: pd.DataFrame, baseline_window: int = 60) -> pd.DataFrame:
    """Turn level features into *stationary* surprise features.

    Raw ``article_count`` drifts with outlet coverage and with GDELT's own
    indexing over the years, so a tree that learns "count > 80 means high vol"
    in fold 3 is learning a date proxy, not a signal. What is stationary is the
    count relative to its own recent baseline -- a news-volume shock.
    """
    out = daily.sort_values("trading_day").reset_index(drop=True)

    log_count = np.log1p(out["article_count"])
    roll_mu = log_count.rolling(baseline_window, min_periods=10).mean()
    roll_sd = log_count.rolling(baseline_window, min_periods=10).std()
    out["news_vol_z"] = (log_count - roll_mu) / roll_sd.replace(0, np.nan)

    abs_mu = out["sent_abs_mean"].rolling(baseline_window, min_periods=10).mean()
    abs_sd = out["sent_abs_mean"].rolling(baseline_window, min_periods=10).std()
    out["sent_abs_shock"] = (out["sent_abs_mean"] - abs_mu) / abs_sd.replace(0, np.nan)

    out["sent_disp_roll3"] = out["sent_disp"].rolling(3, min_periods=1).mean()
    out["sent_abs_mean_roll5"] = out["sent_abs_mean"].rolling(5, min_periods=1).mean()
    return out


def build_feature_table(prices: pd.DataFrame,
                        scored_news: pd.DataFrame | None = None,
                        vix_close: pd.Series | None = None,
                        horizons: tuple[int, ...] = (1, 5),
                        baseline_window: int = 60) -> pd.DataFrame:
    """Assemble the full daily feature table plus targets for each horizon."""
    table = build_price_features(prices)
    table = add_vix_features(table, vix_close)

    daily = aggregate_daily_sentiment(scored_news)
    if len(daily) > 0:
        daily = add_sentiment_dynamics(daily, baseline_window)
        table = table.merge(
            daily.set_index("trading_day"),
            left_index=True, right_index=True, how="left",
        )
        table["has_news"] = table["article_count"].notna().astype(float)
        # Deliberately NOT filling the sentiment columns: NaN means "no news",
        # which is different information from "neutral news".
    else:
        for col in SENTIMENT_FEATURES:
            table[col] = np.nan
        table["has_news"] = 0.0

    return add_targets(table, horizons)


def add_targets(table: pd.DataFrame, horizons: tuple[int, ...] = (1, 5)) -> pd.DataFrame:
    """Forward realized volatility over the next ``h`` trading days.

    Two targets per horizon, because they are used for different things and
    conflating them is a real source of error:

    ``target_log_rv_h{h}``
        Mean of forward *log*  This is the modelling target -- log-vol is
        near-Gaussian and homoscedastic, which is what a squared-error
        objective wants.

    ``target_rvar_h{h}``
        Mean of forward *variance*, i.e. arithmetic realized variance. This is
        what a variance swap actually settles against, and it is strictly
        larger than the squared geometric mean by Jensen. Using the log-mean
        target to settle the VRP payoff understates realized variance on
        exactly the spiky days that hurt a short-vol book, which flatters the
        strategy precisely where it should be punished.
    """
    out = table.copy()
    var = out["rv"] ** 2
    for h in horizons:
        out[f"target_log_rv_h{h}"] = (
            out["log_rv"].shift(-1).rolling(h, min_periods=h).mean().shift(-(h - 1)))
        out[f"target_rvar_h{h}"] = (
            var.shift(-1).rolling(h, min_periods=h).mean().shift(-(h - 1)))
    return out


def feature_columns(table: pd.DataFrame, use_sentiment: bool = True,
                    use_vix: bool = True) -> list[str]:
    cols = [c for c in PRICE_FEATURES if c in table.columns]
    if use_vix:
        cols += [c for c in VIX_FEATURES if c in table.columns
                 and table[c].notna().any()]
    if use_sentiment:
        cols += [c for c in SENTIMENT_FEATURES if c in table.columns
                 and table[c].notna().any()]
    return cols

In [ ]:
table = build_feature_table(prices, scored_news, vix, horizons=(1, 5))
cols = feature_columns(table)
print(f"{len(cols)} features, {len(table)} rows")
print(cols)
table[["rv", "gk_vol", "iv_daily", "log_vrp", "target_log_rv_h1"]].tail()

In [ ]:
# The v1 bug, measured on your own data rather than assumed.
share = (table["gk_vol"] / table["rv"]).median()
print(f"median sigma_GK / sigma_total = {share:.3f}")
print(f"-> sizing off Garman-Klass over-levers by {1/share:.2f}x")

## 5. Baselines

**HAR-RV** (Corsi 2009) is the bar that matters, not naive persistence.
`log(sigma_t)` from a single day's range is a very noisy level estimate, so
v1's "+15% vs naive" is close to free — a 5-day moving average captures most
of it with no model.

GARCH is also fixed: v1 fit close-to-close returns (forecasting *total* vol)
and scored it against a Garman-Klass *intraday* target, a near-constant log
offset of ~+0.21 that inflated its RMSE. Most of "+30.8% vs GARCH" was that
unit error.

In [ ]:
# ===== src/baselines.py =====
"""Baselines the model has to beat -- and the reason v1's margins were inflated.

v1 reported "+15.0% RMSE vs naive" and "+30.8% vs GARCH". Both numbers are
softer than they look:

* **Naive persistence** is ``log(sigma_t)`` from a *single day's* range. As an
  estimator of the current vol level it is extremely noisy, so beating it by
  15% is close to free -- a 5-day moving average of the same quantity does most
  of it. It is a sanity check, not a competitor.

* **GARCH was scored in the wrong units.** ``fit_garch_forecast`` fits
  close-to-close returns, so it forecasts *total* (gap-inclusive) vol, but it
  was scored against a Garman-Klass *intraday* target. On a log scale that is a
  near-constant offset of ``log(sigma_total / sigma_GK) ~ +0.2``, which inflates
  GARCH's RMSE by roughly that amount regardless of how good the model is. Most
  of the "+30.8%" is a unit error, not skill.

The honest bar for a daily realized-volatility forecast is **HAR-RV**
(Corsi 2009): an OLS regression of tomorrow's log-RV on daily, weekly and
monthly averages of log-RV. It is three lines of code, has no hyperparameters,
and is genuinely hard to beat. If LightGBM plus FinBERT cannot beat HAR, the
project's headline claim does not survive.
"""


import numpy as np
import pandas as pd

HAR_LAGS = {"d": 1, "w": 5, "m": 22}


# --------------------------------------------------------------------------
# naive persistence
# --------------------------------------------------------------------------

def naive_forecast(log_rv: pd.Series) -> pd.Series:
    """Tomorrow's log-vol = today's log-"""
    return log_rv.copy()


# --------------------------------------------------------------------------
# HAR-RV
# --------------------------------------------------------------------------

def har_design(log_rv: pd.Series) -> pd.DataFrame:
    """HAR regressors at date t (all known at t's close)."""
    return pd.DataFrame({
        "har_d": log_rv,
        "har_w": log_rv.rolling(HAR_LAGS["w"]).mean(),
        "har_m": log_rv.rolling(HAR_LAGS["m"]).mean(),
    }, index=log_rv.index)


class HARModel:
    """OLS HAR-RV, fit by least squares with an intercept."""

    def __init__(self) -> None:
        self.coef_: np.ndarray | None = None
        self.columns_ = ["har_d", "har_w", "har_m"]

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "HARModel":
        design = X[self.columns_]
        mask = design.notna().all(axis=1) & y.notna()
        A = np.column_stack([np.ones(mask.sum()), design.loc[mask].to_numpy()])
        self.coef_, *_ = np.linalg.lstsq(A, y.loc[mask].to_numpy(), rcond=None)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        if self.coef_ is None:
            raise RuntimeError("HARModel.fit must be called before predict")
        design = X[self.columns_].to_numpy()
        A = np.column_stack([np.ones(len(design)), design])
        # Rows with a NaN regressor fall back to the intercept-only prediction.
        out = A @ self.coef_
        bad = ~np.isfinite(A).all(axis=1)
        out[bad] = self.coef_[0]
        return out


# --------------------------------------------------------------------------
# GARCH(1,1)
# --------------------------------------------------------------------------

def garch_forecast(log_returns: pd.Series, refit_every: int = 5,
                   min_history: int = 250) -> pd.Series:
    """Rolling one-step-ahead GARCH(1,1) forecast of *total* daily log-

    Indexed so that the value at date t is the forecast for t+1 -- the same
    convention as every other forecast in this project, so no caller-side
    ``shift`` is required (v1 pushed that onto the caller, which is exactly the
    kind of thing that silently goes wrong).

    Returns log-vol in the same close-to-close units as ``target_log_rv_h1``,
    so the comparison is now apples to apples.
    """
    from arch import arch_model

    rets = (log_returns.dropna() * 100.0)
    out = pd.Series(index=rets.index, dtype=float)

    last_params = None
    for i in range(min_history, len(rets)):
        history = rets.iloc[:i]
        try:
            model = arch_model(history, vol="GARCH", p=1, q=1, mean="Zero",
                               rescale=False)
            if last_params is None or (i - min_history) % refit_every == 0:
                res = model.fit(disp="off")
                last_params = res.params
            else:
                res = model.fix(last_params)
            var = res.forecast(horizon=1, reindex=False).variance.values[-1, 0]
            # forecast made from data strictly before rets.index[i] is a
            # forecast *for* rets.index[i]; store it at i-1 so the index
            # convention is "value at t forecasts t+1".
            out.iloc[i - 1] = np.log(np.sqrt(var) / 100.0)
        except Exception:
            continue

    return out.reindex(log_returns.index)


def garch_bias_correction(garch_log: pd.Series, target_log: pd.Series,
                          train_mask: np.ndarray) -> float:
    """Mean log offset between GARCH and the target, estimated on training data.

    Even with matched units a one-observation realized-vol target sits below a
    conditional-vol forecast in expectation (Jensen, plus estimator noise). We
    remove that offset using training data only, so GARCH is scored on its
    *shape*, which is what a baseline comparison is supposed to test.
    """
    a = garch_log[train_mask]
    b = target_log[train_mask]
    mask = a.notna() & b.notna()
    if mask.sum() < 30:
        return 0.0
    return float((b[mask] - a[mask]).mean())

## 6. Model

One real bug fixed — see the docstring.

In [ ]:
# ===== src/model.py =====
"""LightGBM volatility model.

One real bug fixed from v1. v1 tuned hyperparameters with early stopping on a
validation tail, took ``n_estimators`` straight out of the trial's *suggested*
value, and then refit on the full training window **with no early stopping**:

    best_params = tune_hyperparameters(X_tr, y_tr, X_val, y_val, ...)
    model = train_lgbm(train_df[feature_cols], train_df[target], params=best_params)

So the number of trees that survived early stopping during tuning was thrown
away, and the refit ran the full suggested ``n_estimators`` -- up to 500 -- on
data it had never been validated against. The tuned configuration and the
deployed model were not the same model. Here the surviving iteration count is
carried through and rescaled for the larger refit sample.
"""


import numpy as np
import pandas as pd

RANDOM_SEED = 42

BASE_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "seed": RANDOM_SEED,
    "n_jobs": -1,
}


def _search_space(trial) -> dict:
    return {
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": 2000,  # capped by early stopping, not by the search
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }


def tune(X_tr, y_tr, X_val, y_val, n_trials: int = 25) -> dict:
    """Optuna search; returns params including the early-stopped tree count."""
    import lightgbm as lgb
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    best_iters: dict[int, int] = {}

    def objective(trial):
        params = {**BASE_PARAMS, **_search_space(trial)}
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        best_iters[trial.number] = int(model.best_iteration_ or params["n_estimators"])
        pred = model.predict(X_val, num_iteration=model.best_iteration_)
        return float(np.sqrt(np.mean((pred - np.asarray(y_val)) ** 2)))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    params = {**BASE_PARAMS, **_search_space_from_dict(study.best_params)}
    params["n_estimators"] = best_iters.get(study.best_trial.number, 200)
    return params


def _search_space_from_dict(d: dict) -> dict:
    out = dict(d)
    out["subsample_freq"] = 1
    return out


def fit(X, y, params: dict, val_fraction: float = 0.0):
    """Fit LightGBM, rescaling the tree count for the larger refit sample.

    ``params['n_estimators']`` arrives as the count that survived early stopping
    on ``val_fraction``-smaller data; scaling it up keeps the effective amount
    of boosting roughly constant on the full window.
    """
    import lightgbm as lgb

    p = dict(params)
    if val_fraction > 0:
        p["n_estimators"] = max(20, int(p.get("n_estimators", 200) / (1 - val_fraction)))
    model = lgb.LGBMRegressor(**p)
    model.fit(X, y)
    return model


def fit_quantiles(X, y, params: dict, quantiles=(0.1, 0.5, 0.9),
                  val_fraction: float = 0.0) -> dict:
    """One model per quantile for a cheap predictive band."""
    models = {}
    for q in quantiles:
        p = dict(params)
        p["objective"] = "quantile"
        p["alpha"] = q
        p.pop("metric", None)
        models[q] = fit(X, y, p, val_fraction)
    return models

## 7. Walk-forward validation

Adds a **Diebold-Mariano test** with Newey-West standard errors, so "the model
beats HAR" becomes a claim with a p-value rather than a bare percentage over
24 noisy folds. Also **purges** overlapping multi-day targets at the
train/test boundary, without which the `h=5` model trains on labels built from
the data it is about to be scored on.

In [ ]:
# ===== src/validation.py =====
"""Walk-forward validation, with a significance test instead of a bare percentage.

v1's headline was "+15.0% RMSE vs naive". Across 24 folds of a noisy daily
series that number carries no error bar, and a percentage improvement over a
deliberately weak baseline is the easiest kind of result to produce by
accident. This module adds:

  * **HAR-RV** as the baseline that actually matters (see src/py);
  * a **Diebold-Mariano test** on the squared-error differential, with
    Newey-West standard errors, so "the model beats HAR" becomes a claim with a
    p-value attached;
  * per-fold smearing factors estimated on training residuals, so the level
    forecasts handed to the backtest are unbiased in the mean rather than the
    median (this is the correction v1 omitted -- see volatility.smearing_factor).
"""


import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta



@dataclass
class Fold:
    fold: int
    train_start: pd.Timestamp
    train_end: pd.Timestamp
    test_start: pd.Timestamp
    test_end: pd.Timestamp
    n_train: int
    n_test: int
    smearing: float
    predictions: pd.DataFrame = field(repr=False)


def generate_folds(dates: pd.DatetimeIndex, train_years: int = 2,
                   test_months: int = 2, step_months: int = 2,
                   purge: int = 0):
    """Expanding-window folds. Never a shuffle split.

    ``purge`` drops the last ``purge`` rows of each training window. With a
    multi-day target this is mandatory: the target at the final training date
    is an average over days that fall inside the test window, so without
    purging the model is trained on labels built from the data it is about to
    be scored on. For ``target_*_h{h}`` the correct value is ``h - 1``.
    """
    dates = pd.DatetimeIndex(dates)
    start = dates.min()
    train_end = start + relativedelta(years=train_years)
    while True:
        test_end = train_end + relativedelta(months=test_months)
        train_mask = (dates >= start) & (dates < train_end)
        test_mask = (dates >= train_end) & (dates < test_end)
        if purge > 0 and train_mask.sum() > purge:
            keep = np.flatnonzero(train_mask)[:-purge]
            purged = np.zeros_like(train_mask)
            purged[keep] = True
            train_mask = purged
        if test_mask.sum() == 0:
            break
        if train_mask.sum() > 0:
            yield train_mask, test_mask
        train_end = train_end + relativedelta(months=step_months)
        if train_end >= dates.max():
            break


def run_walk_forward(table: pd.DataFrame, feature_cols: list[str],
                     target_col: str = "target_log_rv_h1",
                     garch_log: pd.Series | None = None,
                     train_years: int = 2, test_months: int = 2,
                     step_months: int = 2, n_trials: int = 25,
                     retune_every: int = 4, val_fraction: float = 0.15,
                     purge: int | None = None,
                     verbose: bool = True) -> list[Fold]:
    """Expanding walk-forward over LightGBM, HAR, naive and (optionally) GARCH.

    ``retune_every`` re-runs Optuna only every k folds and reuses the params in
    between. v1 ran a fresh 20-trial study on all 24 folds, which is both slow
    and a source of fold-to-fold noise that has nothing to do with the signal.

    ``purge`` defaults to ``h - 1`` inferred from ``target_col``, which is what
    a multi-day overlapping target requires. See generate_folds.
    """
    if purge is None:
        m = re.search(r"_h(\d+)$", target_col)
        purge = (int(m.group(1)) - 1) if m else 0
    df = table.dropna(subset=[target_col]).copy()
    dates = df.index
    log_rv = df["log_rv"]
    har_X = har_design(log_rv)

    folds: list[Fold] = []
    params: dict | None = None

    for i, (train_mask, test_mask) in enumerate(
            generate_folds(dates, train_years, test_months, step_months, purge)):
        train_df, test_df = df.loc[train_mask], df.loc[test_mask]
        if len(train_df) < 120 or len(test_df) == 0:
            continue

        X_tr_all, y_tr_all = train_df[feature_cols], train_df[target_col]
        cut = int(len(train_df) * (1 - val_fraction))
        X_tr, y_tr = X_tr_all.iloc[:cut], y_tr_all.iloc[:cut]
        X_val, y_val = X_tr_all.iloc[cut:], y_tr_all.iloc[cut:]

        if params is None or i % retune_every == 0:
            params = tune(X_tr, y_tr, X_val, y_val, n_trials=n_trials)

        lgbm = fit(X_tr_all, y_tr_all, params, val_fraction=val_fraction)

        # Smearing estimated on the held-out validation tail of the *training*
        # window -- in-sample residuals would understate it badly.
        val_resid = np.asarray(y_val) - lgbm.predict(X_val)
        smear = smearing_factor(val_resid)

        har = HARModel().fit(har_X.loc[train_mask], y_tr_all)

        preds = pd.DataFrame({
            "date": test_df.index,
            "y_true": test_df[target_col].to_numpy(),
            "model": lgbm.predict(test_df[feature_cols]),
            "naive": naive_forecast(log_rv.loc[test_mask]).to_numpy(),
            "har": har.predict(har_X.loc[test_mask]),
        })

        if garch_log is not None:
            offset = garch_bias_correction(
                garch_log.reindex(df.index), df[target_col], train_mask)
            preds["garch"] = garch_log.reindex(test_df.index).to_numpy() + offset
        else:
            preds["garch"] = np.nan

        folds.append(Fold(
            fold=len(folds),
            train_start=train_df.index.min(), train_end=train_df.index.max(),
            test_start=test_df.index.min(), test_end=test_df.index.max(),
            n_train=len(train_df), n_test=len(test_df),
            smearing=smear, predictions=preds,
        ))
        if verbose:
            print(f"  fold {folds[-1].fold:2d}  "
                  f"test {preds['date'].min().date()} -> {preds['date'].max().date()}  "
                  f"n={len(test_df):3d}  smearing={smear:.3f}")

    return folds


# --------------------------------------------------------------------------
# metrics
# --------------------------------------------------------------------------

def stack_predictions(folds: list[Fold]) -> pd.DataFrame:
    out = pd.concat([f.predictions for f in folds], ignore_index=True)
    smear = pd.concat([
        pd.Series(f.smearing, index=range(len(f.predictions))) for f in folds
    ], ignore_index=True)
    out["smearing"] = smear.to_numpy()
    return out.set_index("date").sort_index()


def error_table(preds: pd.DataFrame,
                cols=("model", "har", "naive", "garch")) -> pd.DataFrame:
    rows = []
    for c in cols:
        if c not in preds or preds[c].isna().all():
            continue
        mask = preds[c].notna() & preds["y_true"].notna()
        err = preds.loc[mask, c] - preds.loc[mask, "y_true"]
        rows.append({
            "forecast": c,
            "rmse": float(np.sqrt((err**2).mean())),
            "mae": float(err.abs().mean()),
            "n": int(mask.sum()),
        })
    return pd.DataFrame(rows).set_index("forecast")


def diebold_mariano(preds: pd.DataFrame, a: str = "model", b: str = "har",
                    lag: int | None = None) -> dict:
    """DM test on the squared-error differential ``e_a^2 - e_b^2``.

    Negative statistic => forecast ``a`` has the lower loss. Newey-West
    standard errors, since daily volatility errors are serially correlated and
    a naive t-stat would overstate significance.
    """
    mask = preds[a].notna() & preds[b].notna() & preds["y_true"].notna()
    ea = (preds.loc[mask, a] - preds.loc[mask, "y_true"]).to_numpy()
    eb = (preds.loc[mask, b] - preds.loc[mask, "y_true"]).to_numpy()
    d = ea**2 - eb**2
    T = len(d)
    if T < 30:
        return {"statistic": np.nan, "p_value": np.nan, "n": T}

    if lag is None:
        lag = int(np.floor(4 * (T / 100.0) ** (2.0 / 9.0)))
    dbar = d.mean()
    dc = d - dbar
    gamma0 = float(dc @ dc / T)
    var = gamma0
    for k in range(1, lag + 1):
        gk = float(dc[k:] @ dc[:-k] / T)
        var += 2.0 * (1.0 - k / (lag + 1.0)) * gk
    var = max(var, 1e-18)

    stat = dbar / np.sqrt(var / T)
    # two-sided normal p-value
    from math import erfc, sqrt
    p = erfc(abs(stat) / sqrt(2.0))
    return {"statistic": float(stat), "p_value": float(p), "n": T,
            "mean_loss_diff": float(dbar), "lag": lag}

In [ ]:
folds = run_walk_forward(table, cols, target_col="target_log_rv_h1",
                          train_years=2, test_months=2, step_months=2,
                          n_trials=25)
preds = stack_predictions(folds)
errs = error_table(preds)
print(errs)

base = errs.loc["har", "rmse"]
print(f"\nModel vs HAR-RV: {100*(base-errs.loc['model','rmse'])/base:+.1f}% RMSE")
dm = diebold_mariano(preds, "model", "har")
print(f"Diebold-Mariano: stat={dm['statistic']:+.2f}  p={dm['p_value']:.4f}")
print("->", "significant" if dm["p_value"] < 0.05 and dm["statistic"] < 0
      else "NOT significant -- the model does not beat HAR")

In [ ]:
folds5 = run_walk_forward(table, cols, target_col="target_log_rv_h5",
                           train_years=2, n_trials=25, verbose=False)
preds5 = stack_predictions(folds5)
print(error_table(preds5))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(preds.index, np.exp(preds["y_true"]) * np.sqrt(252), lw=1.1, label="Realized")
ax.plot(preds.index, np.exp(preds["model"]) * preds["smearing"] * np.sqrt(252),
        lw=1.1, label="LightGBM")
ax.plot(table["iv_daily"].reindex(preds.index) * np.sqrt(252), lw=0.9,
        alpha=0.7, label="India VIX (implied)")
ax.set_title("Walk-forward forecast vs realized, annualized")
ax.legend(); ax.grid(alpha=0.3); plt.show()

## 8. Explainability — and a better question than SHAP share

v1 reported "sentiment contributes 0.0% of total feature importance" and
stopped. But SHAP share describes *this fitted model*, not the data: when
sentiment features are noisy and partly collinear with price features, greedy
tree splitting simply never selects them and their SHAP mass is zero by
construction.

The decision-useful question is whether out-of-sample error gets worse when
you remove them. `incremental_value` runs that ablation and attaches a
p-value. It may still come back negative — that is a respectable finding, and
a tested negative beats an untestable SHAP share.

In [ ]:
# ===== src/explain.py =====
"""SHAP attribution, and a fairer test of whether sentiment contributes.

v1 reported "sentiment contributes 0.0% of total feature importance" and left
it there. Two things are wrong with stopping at that number.

First, |SHAP| share is a statement about *this fitted model*, not about the
data. When sentiment features are collinear with price features and arrive
second in the tree-building order, LightGBM will simply never split on them and
their SHAP mass is zero by construction. That is a fact about greedy splitting,
not evidence that news carries no information.

Second, the decision-useful question is not "what share of importance" but
"does out-of-sample error get worse if I remove these features". That is a
one-line experiment and it is the one that settles the argument.
``incremental_value`` runs it.
"""


import numpy as np
import pandas as pd

SENTIMENT_PREFIXES = ("sent_", "news_vol", "article_count", "pct_", "has_news")
VIX_PREFIXES = ("log_iv", "log_vrp", "vix_")


def compute_shap(model, X: pd.DataFrame):
    import shap
    return shap.TreeExplainer(model)(X)


def importance(shap_values, feature_names: list[str]) -> pd.DataFrame:
    mean_abs = np.abs(shap_values.values).mean(axis=0)
    return (pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True))


def _group(name: str) -> str:
    if name.startswith(SENTIMENT_PREFIXES):
        return "sentiment"
    if name.startswith(VIX_PREFIXES):
        return "implied_vol"
    return "price"


def group_shares(importance_df: pd.DataFrame) -> pd.Series:
    g = importance_df.assign(group=importance_df["feature"].map(_group))
    total = g["mean_abs_shap"].sum()
    if total == 0:
        return pd.Series(dtype=float)
    return (100 * g.groupby("group")["mean_abs_shap"].sum() / total
            ).sort_values(ascending=False)


def incremental_value(table: pd.DataFrame, base_cols: list[str],
                      extra_cols: list[str], target_col: str = "target_log_rv_h1",
                      **wf_kwargs) -> dict:
    """Does adding ``extra_cols`` reduce walk-forward RMSE? With a p-value.

    This is the honest replacement for "sentiment contributes 0.0% of SHAP".
    Runs the same walk-forward twice -- once without the extra block, once with
    it -- and Diebold-Mariano tests the difference in squared error. If the
    p-value is not small, the extra features do not help, and that is a real
    finding worth reporting rather than something to hide.
    """

    wf_kwargs.setdefault("verbose", False)
    base = stack_predictions(
        run_walk_forward(table, base_cols, target_col, **wf_kwargs))
    full = stack_predictions(
        run_walk_forward(table, base_cols + extra_cols, target_col,
                                    **wf_kwargs))

    joined = pd.DataFrame({
        "y_true": base["y_true"],
        "base": base["model"],
        "full": full["model"].reindex(base.index),
    }).dropna()

    rmse_base = float(np.sqrt(((joined["base"] - joined["y_true"]) ** 2).mean()))
    rmse_full = float(np.sqrt(((joined["full"] - joined["y_true"]) ** 2).mean()))
    dm = diebold_mariano(joined, "full", "base")

    return {
        "rmse_without": rmse_base,
        "rmse_with": rmse_full,
        "improvement_pct": 100 * (rmse_base - rmse_full) / rmse_base,
        "dm_statistic": dm["statistic"],
        "p_value": dm["p_value"],
        "n": dm["n"],
        "verdict": ("adds real information" if dm["statistic"] < 0 and dm["p_value"] < 0.05
                    else "no significant contribution"),
    }

In [ ]:
final = table.dropna(subset=["target_log_rv_h1"])
cut = int(len(final) * 0.85)
params = tune(final[cols].iloc[:cut], final["target_log_rv_h1"].iloc[:cut],
              final[cols].iloc[cut:], final["target_log_rv_h1"].iloc[cut:], n_trials=30)
final_model = fit(final[cols], final["target_log_rv_h1"], params, val_fraction=0.15)

imp = importance(compute_shap(final_model, final[cols].iloc[-500:]), cols)
print(group_shares(imp).round(1).to_string(), "\n")
print(imp.head(12).to_string(index=False))

In [ ]:
sent_cols = [c for c in cols if c.startswith(("sent_", "news_vol", "has_news"))]
if sent_cols:
    base_cols = [c for c in cols if c not in sent_cols]
    print(incremental_value(table, base_cols, sent_cols, n_trials=10, train_years=2))
else:
    print("No sentiment features wired in -- see section 3.")

## 9. Backtest 1 — volatility-targeted index exposure

v1's idea, repaired: correct units, smearing applied, financing charged on
borrowed exposure and cash credited when under-invested, and a no-trade band
with partial adjustment instead of chasing a noisy daily target.

**This will not make money and is not supposed to.** Sharpe is invariant to
leverage, so the only thing that moves it is `Cov(1/sigma_hat, r_next)` — and
the model forecasts volatility, not returns. Treat it as a risk overlay and
claim the drawdown, not the return.

In [ ]:
# ===== src/backtest.py =====
"""Backtests.

Two strategies, for two different reasons.

``vol_target_backtest`` is v1's idea, repaired. It will not make money and it
is not supposed to -- a long-only position sized by a volatility forecast has
no expected-return edge, because Sharpe is invariant to leverage. The most it
can do is reshape risk: shallower drawdowns for a similar return. v1 reported
it as a profit strategy and lost to buy-and-hold on every axis; fixed, it
should land at a modestly better Sharpe and a much better drawdown, with the
honest caveat attached.

``vrp_backtest`` is where a volatility forecast actually pays. India VIX prices
*implied* volatility; the model forecasts *realized* volatility; the spread
between them is the variance risk premium, which on the Nifty is large and
persistently positive. Selling variance harvests it, and the model's job is to
tell you when the premium is unusually rich (size up) or thin/negative (stand
aside, or buy). This converts the volatility forecast from a position-sizing
input, where its skill is nearly worthless, into a pricing input, where its
skill is the whole trade.

The short-variance leg has severe negative skew. Everything here is capped and
limited, and the caveats are in CRITICAL_ANALYSIS.md section 5 -- read them
before believing any Sharpe this file prints.
"""


import numpy as np
import pandas as pd

TRADING_DAYS = 252


# --------------------------------------------------------------------------
# shared metrics
# --------------------------------------------------------------------------

def sharpe(returns: pd.Series, rf_annual: float = 0.0) -> float:
    r = pd.Series(returns).dropna()
    if len(r) < 2 or r.std() == 0:
        return 0.0
    excess = r - rf_annual / TRADING_DAYS
    return float(np.sqrt(TRADING_DAYS) * excess.mean() / r.std())


def max_drawdown(equity: pd.Series) -> float:
    eq = pd.Series(equity).dropna()
    if eq.empty:
        return 0.0
    return float((eq / eq.cummax() - 1.0).min())


def summarize(returns: pd.Series, rf_annual: float = 0.0,
              label: str = "") -> dict:
    r = pd.Series(returns).dropna()
    equity = (1 + r).cumprod()
    years = len(r) / TRADING_DAYS
    cagr = float(equity.iloc[-1] ** (1 / years) - 1) if years > 0 and len(equity) else 0.0

    # Sortino must use the same excess return as Sharpe. Comparing an
    # excess-return Sharpe against a raw-return Sortino produces contradictory
    # signs on the same series and makes a cash-heavy book look good.
    excess = r - rf_annual / TRADING_DAYS
    downside = excess[excess < 0].std()
    return {
        "label": label,
        "cagr": cagr,
        "ann_vol": float(r.std() * np.sqrt(TRADING_DAYS)),
        "sharpe": sharpe(r, rf_annual),
        "sortino": float(np.sqrt(TRADING_DAYS) * excess.mean() / downside) if downside else 0.0,
        "max_drawdown": max_drawdown(equity),
        "total_return": float(equity.iloc[-1] - 1) if len(equity) else 0.0,
        "skew": float(r.skew()),
        "worst_day": float(r.min()),
    }


def _apply_deadband(raw: pd.Series, deadband: float) -> pd.Series:
    """No-trade band with partial adjustment to the band edge.

    v1 rebalanced to a fresh target every single day off a noisy daily forecast
    and paid 7.5 bps on every wiggle. Backing out v1's own reported numbers,
    that cost drag was ~2-3.5%/yr -- larger than any plausible vol-timing
    benefit, and the single biggest reason the strategy lost to buy-and-hold.

    Snapping to the full target whenever the band is breached barely helps,
    because a noisy target breaches it most days. Trading only to the *edge* of
    the band is the classic no-trade-region result (Leland; Davis-Norman): the
    book tracks the target with a lag, turnover collapses, and the tracking
    error costs far less than the spread it saves.
    """
    if deadband <= 0:
        return raw.ffill().fillna(0.0)

    out = np.empty(len(raw))
    current = float(raw.iloc[0]) if len(raw) else 0.0
    for i, target in enumerate(raw.to_numpy()):
        if np.isfinite(target):
            if target > current + deadband:
                current = float(target) - deadband
            elif target < current - deadband:
                current = float(target) + deadband
        out[i] = current
    return pd.Series(out, index=raw.index)


# --------------------------------------------------------------------------
# strategy 1: volatility-targeted index exposure (the repaired v1 idea)
# --------------------------------------------------------------------------

def vol_target_backtest(pred_vol: pd.Series, next_return: pd.Series,
                        target_ann_vol: float = 0.12,
                        max_leverage: float = 1.5,
                        cost_bps: float = 7.5,
                        rf_annual: float = 0.065,
                        deadband: float = 0.10) -> pd.DataFrame:
    """Long-only index exposure scaled inversely to predicted volatility.

    ``pred_vol`` must be a **gap-inclusive daily** vol forecast with the
    smearing correction already applied -- the same units as the close-to-close
    returns in ``next_return``. Getting this wrong is what produced v1's
    persistent 1.3x leverage.

    Unlike v1 this charges financing on borrowed exposure and credits cash on
    un-invested capital. v1 handed the strategy free leverage, which flatters
    it, and it still lost.
    """
    idx = pred_vol.index.intersection(next_return.index)
    pv = pred_vol.loc[idx].astype(float)
    ret = next_return.loc[idx].astype(float)

    target_daily = target_ann_vol / np.sqrt(TRADING_DAYS)
    raw = (target_daily / pv.replace(0, np.nan)).clip(0.0, max_leverage)
    pos = _apply_deadband(raw.ffill().fillna(0.0), deadband)

    traded = pos.diff().abs().fillna(pos.abs())
    cost = traded * (cost_bps / 1e4)
    # borrow above 1x, earn cash below 1x
    financing = (pos - 1.0) * (rf_annual / TRADING_DAYS)

    strat = pos * ret - cost - financing

    out = pd.DataFrame({
        "predicted_vol": pv,
        "position": pos,
        "target_position": raw,
        "next_return": ret,
        "cost": cost,
        "financing": financing,
        "strategy_return": strat,
        "buy_hold_return": ret,
    })
    out["strategy_equity"] = (1 + out["strategy_return"]).cumprod()
    out["buy_hold_equity"] = (1 + out["buy_hold_return"]).cumprod()
    return out


def futures_carry(position: pd.Series, rf_annual: float = 0.065,
                  div_yield: float = 0.012) -> pd.Series:
    """Daily carry for a fully-collateralised Nifty futures position.

    You cannot short the cash index in India; shorting means futures. That
    changes the financing arithmetic and it is worth getting right rather than
    bolting a minus sign onto the long-only version.

    A futures price is ``F = S * exp((r - q) * T)``, so holding the future to
    expiry earns the *price* return minus ``(r - q)``. Meanwhile the cash that
    is not posted as margin earns ``r``. Netting the two:

        carry = position * q / 252  +  (1 - position) * r / 252

    Read off the cases: at ``position = 1`` you earn the dividend yield on top
    of the price return, which correctly reconstructs the total return of the
    index (``^NSEI`` is a price index and excludes dividends). At
    ``position = 0`` you earn the risk-free rate on all of it. At
    ``position = -1`` you pay the dividend yield away on the short and collect
    ``r`` on both your own capital and the short proceeds.

    That last line is the one that matters for shorting: the carry is a
    *headwind*, because you are giving up the dividend yield and fighting the
    index's positive drift. A short has to overcome roughly ``q + drift``
    before it breaks even, which is why the long/short version below usually
    looks worse than long-only unless the directional signal is genuinely good.
    """
    return position * (div_yield / TRADING_DAYS) + (1 - position) * (rf_annual / TRADING_DAYS)


def long_short_backtest(direction: pd.Series, pred_vol: pd.Series,
                        next_return: pd.Series,
                        target_ann_vol: float = 0.12,
                        max_long: float = 1.5, max_short: float = -1.0,
                        cost_bps: float = 5.0,
                        rf_annual: float = 0.065, div_yield: float = 0.012,
                        deadband: float = 0.10,
                        stop_drawdown: float | None = -0.25) -> pd.DataFrame:
    """Long/short index exposure: direction from signals, size from the vol model.

    ``position = direction * (target_vol / predicted_vol)``, so the two models
    do separate jobs -- the directional signal picks the side, the volatility
    forecast decides how much risk that side is worth. This is the only
    arrangement in which a volatility forecast contributes to *return* rather
    than only to risk, and it does so indirectly: it makes each unit of
    directional conviction carry constant risk.

    ``cost_bps`` defaults to 5 rather than the 7.5 used for the cash overlay:
    Nifty futures are cheaper to trade (STT on the sell side is 2 bps, spread
    and brokerage roughly another 1-3).

    ``stop_drawdown`` flattens the book if equity falls that far below its high
    water mark, re-entering when the signal next flips. A short book without a
    stop is how accounts get closed; losses on a short are unbounded and margin
    calls arrive at the worst possible moment.
    """
    idx = (direction.index.intersection(pred_vol.index)
           .intersection(next_return.index))
    d = direction.loc[idx].astype(float)
    pv = pred_vol.loc[idx].astype(float)
    ret = next_return.loc[idx].astype(float)

    target_daily = target_ann_vol / np.sqrt(TRADING_DAYS)
    scalar = (target_daily / pv.replace(0, np.nan)).clip(0.0, abs(max_long))
    raw = (d * scalar).clip(max_short, max_long)
    pos = _apply_deadband(raw.ffill().fillna(0.0), deadband)

    if stop_drawdown is not None:
        pos = _apply_drawdown_stop(pos, ret, stop_drawdown, rf_annual, div_yield,
                                   cost_bps)

    traded = pos.diff().abs().fillna(pos.abs())
    cost = traded * (cost_bps / 1e4)
    carry = futures_carry(pos, rf_annual, div_yield)

    strat = pos * ret + carry - cost

    out = pd.DataFrame({
        "direction": d,
        "vol_scalar": scalar,
        "position": pos,
        "next_return": ret,
        "carry": carry,
        "cost": cost,
        "strategy_return": strat,
        "buy_hold_return": ret + div_yield / TRADING_DAYS,
    })
    out["strategy_equity"] = (1 + out["strategy_return"]).cumprod()
    out["buy_hold_equity"] = (1 + out["buy_hold_return"]).cumprod()
    out["gross_exposure"] = pos.abs()
    out["is_short"] = (pos < 0).astype(int)
    return out


def _apply_drawdown_stop(pos: pd.Series, ret: pd.Series, limit: float,
                         rf_annual: float, div_yield: float,
                         cost_bps: float) -> pd.Series:
    """Flatten the book while equity is more than ``limit`` below its peak.

    Walked forward one day at a time because the stop depends on the equity
    curve the stop itself produces -- deriving it from the unstopped curve
    would be a lookahead.

    Re-entry is deliberately conservative: once halted, the book stays flat
    until the target position changes sign relative to what was held when the
    stop fired. Re-entering on the same side that just lost 25% is how a stop
    becomes a formality.
    """
    out = np.zeros(len(pos))
    p = pos.to_numpy()
    r = np.nan_to_num(ret.to_numpy())

    equity, peak = 1.0, 1.0
    halted = False
    halted_side = 0.0
    prev = 0.0

    for i in range(len(p)):
        want = p[i]
        if halted:
            # only wake up when the signal has flipped to the other side
            if halted_side != 0.0 and np.sign(want) == -np.sign(halted_side):
                halted = False
            else:
                want = 0.0

        traded = abs(want - prev)
        carry = (want * (div_yield / TRADING_DAYS)
                 + (1 - want) * (rf_annual / TRADING_DAYS))
        day = want * r[i] + carry - traded * (cost_bps / 1e4)

        equity *= (1 + day)
        peak = max(peak, equity)
        out[i] = want
        prev = want

        if not halted and equity / peak - 1.0 <= limit:
            halted = True
            halted_side = np.sign(want) if want != 0 else np.sign(p[i])

    return pd.Series(out, index=pos.index)


def vol_matched(returns: pd.Series, benchmark: pd.Series) -> pd.Series:
    """Rescale ``returns`` to the benchmark's realized 

    The only fair way to compare total return between a levered and an
    unlevered book. v1 compared a ~1.3x-levered strategy's return against 1x
    buy-and-hold and presented the shortfall as a strategy result.
    """
    a, b = returns.dropna(), benchmark.dropna()
    if a.std() == 0:
        return returns
    return returns * (b.std() / a.std())


# --------------------------------------------------------------------------
# strategy 2: variance risk premium carry (where the forecast actually pays)
# --------------------------------------------------------------------------

def vrp_signal(iv_daily: pd.Series, pred_rv_daily: pd.Series) -> pd.Series:
    """Log richness of implied vol over the model's realized-vol forecast.

    Positive => the market is charging more for volatility than the model
    expects to be delivered => selling variance is favourably priced.
    """
    idx = iv_daily.index.intersection(pred_rv_daily.index)
    return np.log(iv_daily.loc[idx]) - np.log(pred_rv_daily.loc[idx])


def vrp_backtest(iv_daily: pd.Series, pred_rv_daily: pd.Series,
                 realized_var_fwd: pd.Series,
                 entry_z: float = 0.0, max_position: float = 1.0,
                 scale: float = 3.0,
                 allow_long_vol: bool = True,
                 hold_days: int = 5,
                 cost_vol_pts: float = 0.005,
                 risk_fraction: float = 0.15,
                 cap_multiple: float = 3.0) -> pd.DataFrame:
    """Short/long variance against the model's realized-vol forecast.

    Payoff is the standard variance-swap P&L expressed in vega terms:

        pnl_vol_points = (K^2 - RV^2) / (2K)

    with ``K`` the implied vol struck at entry and ``RV`` the volatility
    actually realized over the holding period, both annualized. ``cap_multiple``
    caps the loss at ``cap_multiple * K`` realized vol, which is what buying
    wings (a strangle overlay instead of a naked short) actually buys you --
    without it the downside is unbounded and the backtest is fiction.

    Positions are held ``hold_days`` (Nifty options expire weekly) rather than
    rebalanced daily, so the bid-ask is paid once per position, not 252 times a
    year.

    Parameters
    ----------
    realized_var_fwd
        Forward-looking: the value at ``t`` must be the *arithmetic mean daily
        variance* over ``(t, t+hold_days]`` -- ``target_rvar_h{hold_days}`` from
        features.add_targets. A variance swap settles on realized variance, not
        on the geometric mean of daily vols.
    cost_vol_pts
        Round-trip bid-ask in annualized vol points (0.005 = 0.5 vol pts).
    risk_fraction
        Position sizing, expressed as the fraction of capital a full-size
        position loses if realized vol runs all the way to the cap. 0.15 means
        the worst case on a maximal position is a 15% loss. Sharpe is invariant
        to this; the equity curve and the drawdown are not.
    """
    idx = (iv_daily.index
           .intersection(pred_rv_daily.index)
           .intersection(realized_var_fwd.index))
    iv = iv_daily.loc[idx].astype(float)
    pred = pred_rv_daily.loc[idx].astype(float)
    rvar = realized_var_fwd.loc[idx].astype(float)

    sig = np.log(iv) - np.log(pred)

    # Short variance when implied is rich, long when cheap, flat in between.
    pos = np.clip((sig - entry_z) * scale, -max_position, max_position)
    if not allow_long_vol:
        pos = pos.clip(lower=0.0)
    pos = pd.Series(pos, index=idx).fillna(0.0)

    # Enter only on rebalance dates; hold the book in between.
    active = pd.Series(0.0, index=idx)
    entry = pd.Series(False, index=idx)
    held = 0.0
    for i in range(len(idx)):
        if i % hold_days == 0:
            held = float(pos.iloc[i])
            entry.iloc[i] = True
        active.iloc[i] = held

    K = iv * np.sqrt(TRADING_DAYS)                          # annualized strike vol
    R = np.sqrt(rvar * TRADING_DAYS).clip(upper=cap_multiple * K)
    pnl_vol_pts = (K**2 - R**2) / (2 * K)                # per unit vega, short variance

    # Cost charged once per entry, on the size actually traded.
    prev = active.shift(hold_days).fillna(0.0)
    traded = (active - prev).abs().where(entry, 0.0)
    cost = traded * cost_vol_pts

    gross = active * pnl_vol_pts
    # Size so that a full position losing all the way to the cap costs
    # `risk_fraction` of capital: max loss per unit vega is
    # ((cap*K)^2 - K^2)/(2K) = K(cap^2 - 1)/2.
    max_loss_per_vega = K * (cap_multiple**2 - 1) / 2
    vega_notional = risk_fraction / max_loss_per_vega
    # A position entered at t pays over (t, t+hold_days]; spread the P&L evenly
    # so the return series is daily and Sharpe is not inflated by lumpiness.
    strat_return = (gross - cost) * vega_notional / hold_days

    out = pd.DataFrame({
        "iv_ann": K,
        "pred_rv_ann": pred * np.sqrt(TRADING_DAYS),
        "realized_rv_ann": np.sqrt(rvar * TRADING_DAYS),
        "realized_rv_ann_capped": R,
        "signal": sig,
        "position": active,
        "pnl_vol_points": gross,
        "cost": cost,
        "strategy_return": strat_return,
    })
    out["strategy_equity"] = (1 + out["strategy_return"]).cumprod()
    return out


def vrp_naive_benchmark(iv_daily: pd.Series, trailing_rv: pd.Series,
                        realized_var_fwd: pd.Series, **kwargs) -> pd.DataFrame:
    """Same trade, but sized off trailing realized vol instead of the model.

    This is the baseline the ML model has to beat *as a trading signal*. Simply
    always-short-variance harvests the premium too; the question is whether the
    forecast adds anything over that, and this is how you find out.
    """
    return vrp_backtest(iv_daily, trailing_rv, realized_var_fwd, **kwargs)


# --------------------------------------------------------------------------
# combining
# --------------------------------------------------------------------------

def combine(returns: dict[str, pd.Series],
            weights: dict[str, float] | None = None) -> pd.Series:
    """Weighted blend of strategy return streams on their shared dates.

    The vol-targeted index leg and the VRP carry leg are close to uncorrelated
    day to day -- one is long equity beta, the other is short a variance
    premium -- so blending them is the cheapest Sharpe improvement available.
    """
    df = pd.DataFrame(returns).dropna()
    if weights is None:
        weights = {k: 1.0 / len(returns) for k in returns}
    w = pd.Series(weights).reindex(df.columns).fillna(0.0)
    return (df * w).sum(axis=1)

In [ ]:
next_ret = table["log_return"].shift(-1)
pred_vol = pd.Series(np.exp(preds["model"]) * preds["smearing"], index=preds.index)

# v1's configuration, for comparison: GK units, no smearing, daily rebalance,
# free leverage.
gk_ratio = (table["gk_vol"] / table["rv"]).reindex(preds.index).median()
pred_vol_v1 = pred_vol * gk_ratio / preds["smearing"]

bt_v1 = vol_target_backtest(pred_vol_v1, next_ret, target_ann_vol=0.16,
                             max_leverage=2.0, rf_annual=0.0, deadband=0.0)
bt_v2 = vol_target_backtest(pred_vol, next_ret, target_ann_vol=0.12,
                             max_leverage=1.5, rf_annual=0.065, deadband=0.10)

print(pd.DataFrame([
    summarize(bt_v1["strategy_return"], label="v1 config"),
    summarize(bt_v2["strategy_return"], label="v2 config"),
    summarize(bt_v2["buy_hold_return"], label="buy & hold"),
]).set_index("label")[["cagr", "ann_vol", "sharpe", "max_drawdown", "total_return"]])

print(f"\nmean position  v1={bt_v1['position'].mean():.2f}x  v2={bt_v2['position'].mean():.2f}x")
print(f"cost drag/yr   v1={252*bt_v1['cost'].mean():.2%}  v2={252*bt_v2['cost'].mean():.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(bt_v1["strategy_equity"], label="v1 config", lw=1.2)
ax.plot(bt_v2["strategy_equity"], label="v2 config", lw=1.2)
ax.plot(bt_v2["buy_hold_equity"], label="Buy & hold", lw=1.2)
ax.set_title("Vol-targeted index exposure, net of costs and financing")
ax.legend(); ax.grid(alpha=0.3); plt.show()

## 10. Backtest 2 — variance risk premium carry

Where the forecast actually pays. India VIX prices *implied* volatility, the
model forecasts *realized* volatility, and the spread between them is the
variance risk premium. Selling variance harvests it; the model's job is to
size the trade by how rich the premium is relative to the forecast.

That converts the volatility model from a position-sizing input — where
section 9 shows its skill is nearly worthless — into a **pricing** input,
where every improvement in forecast RMSE maps onto P&L.

The model must beat **both** benchmarks below. The premium itself is free to
anyone willing to be short vol; timing it is the claim.

In [ ]:
hold = 5
pred_rv5 = pd.Series(np.exp(preds5["model"]) * preds5["smearing"], index=preds5.index)
realized_var = table["target_rvar_h5"].reindex(pred_rv5.index)
iv = table["iv_daily"].reindex(pred_rv5.index)
trailing = np.exp(table["log_rv_w"]).reindex(pred_rv5.index)

ok = iv.notna() & pred_rv5.notna() & realized_var.notna() & trailing.notna()
iv, pred_rv5, realized_var, trailing = iv[ok], pred_rv5[ok], realized_var[ok], trailing[ok]

kw = dict(hold_days=hold, cost_vol_pts=0.005, risk_fraction=0.15, cap_multiple=3.0)
always = vrp_backtest(iv, iv, realized_var, entry_z=-1e9, **kw)
naive_timed = vrp_backtest(iv, trailing, realized_var, **kw)
modelled = vrp_backtest(iv, pred_rv5, realized_var, **kw)

print(pd.DataFrame([
    summarize(always["strategy_return"], label="always short variance"),
    summarize(naive_timed["strategy_return"], label="timed off trailing realized vol"),
    summarize(modelled["strategy_return"], label="timed off the model"),
]).set_index("label")[["cagr", "ann_vol", "sharpe", "sortino", "max_drawdown", "skew", "worst_day"]])

**Before quoting any of this.** Short variance is a negatively-skewed carry
trade: many small gains, rare very large losses. A Sharpe computed over a
sample with no volatility crisis in it is close to meaningless here — check
that your window spans March 2020 and look at the drawdown through it. Report
skew and worst day alongside the Sharpe, and note that `cap_multiple` assumes
you are buying wings, which costs real premium the backtest only approximates.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(modelled["iv_ann"], label="India VIX (implied)", lw=1)
axes[0].plot(modelled["pred_rv_ann"], label="Model forecast RV", lw=1)
axes[0].plot(modelled["realized_rv_ann"], label="Realized RV", lw=0.9, alpha=0.7)
axes[0].set_title("Implied vs forecast vs realized"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(modelled["position"], lw=1, color="tab:purple")
axes[1].axhline(0, color="k", lw=0.6)
axes[1].set_title("Position (+ = short variance)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 11. Blend

The two legs are close to uncorrelated — one is long equity beta, the other is
short a variance premium — so blending them is the cheapest Sharpe improvement
available here.

In [ ]:
blended = combine({"index": bt_v2["strategy_return"], "vrp": modelled["strategy_return"]},
                  {"index": 0.7, "vrp": 0.3})
corr = pd.DataFrame({"a": bt_v2["strategy_return"],
                     "b": modelled["strategy_return"]}).dropna().corr().iloc[0, 1]

print(pd.DataFrame([
    summarize(bt_v2["strategy_return"], label="index leg"),
    summarize(modelled["strategy_return"], label="VRP leg"),
    summarize(blended, label="70/30 blend"),
    summarize(bt_v2["buy_hold_return"], label="buy & hold"),
]).set_index("label")[["cagr", "ann_vol", "sharpe", "sortino", "max_drawdown"]])
print(f"\nleg correlation = {corr:+.3f}")

## 12. What to claim

Claimable:

- forecast error against **HAR-RV** with a DM p-value — not against a
  single-day persistence strawman;
- the vol-target overlay as a **risk overlay**: drawdown reduction at
  comparable return, financing charged, turnover controlled;
- the VRP strategy against **both** the untimed and the trailing-vol-timed
  benchmarks;
- sentiment's contribution as an ablation with a p-value, whichever way it
  goes.

Not claimable:

- that vol targeting beats buy-and-hold on return — section 9 explains why it
  structurally cannot;
- a short-variance Sharpe from a sample with no volatility crisis in it;
- anything at all from `simulate_market`.